In [1]:
import os, sys, json, re, time, threading
from datetime import datetime
from pathlib import Path

try:
    import joblib
    import mlflow
    import mlflow.sklearn
    import numpy as np
    import pandas as pd
    import requests
    import uvicorn
    from fastapi import FastAPI
except Exception:
    import sys as _s, subprocess as _sp
    _sp.check_call([_s.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'scikit-learn', 'mlflow', 'fastapi', 'uvicorn', 'joblib', 'requests'])
    import joblib
    import mlflow
    import mlflow.sklearn
    import numpy as np
    import pandas as pd
    import requests
    import uvicorn
    from fastapi import FastAPI

p = Path.cwd()
while not (p / 'churnDataset.csv').exists() and p.parent != p:
    p = p.parent
os.chdir(p)
rt = Path.cwd()
dd = rt / 'data'
md = rt / 'models'
ld = rt / 'logs'
mlr = rt / 'mlruns'
for d in [dd, md, ld]:
    d.mkdir(exist_ok=True)

mlflow.set_tracking_uri(mlr.resolve().as_uri())
app = FastAPI(title='churn api')
mm = None

def cn(x):
    x = str(x).strip().lower()
    x = re.sub(r'[^a-z0-9]+', '_', x)
    return x.strip('_')

def fe(df):
    df = df.copy()
    df['spend_per_tenure'] = df['total_spend'] / df['tenure'].replace(0, np.nan)
    df['call_delay_ratio'] = df['support_calls'] / (df['payment_delay'] + 1)
    df['usage_recent_score'] = df['usage_frequency'] / (df['last_interaction'] + 1)
    df['spend_usage_ratio'] = df['total_spend'] / (df['usage_frequency'] + 1)
    return df.replace([np.inf, -np.inf], np.nan)

def prep(x):
    df = pd.DataFrame([x])
    df.columns = [cn(c) for c in df.columns]
    df = fe(df)
    cm = json.loads((dd / 'columns.json').read_text(encoding='utf-8'))
    for c in cm['features']:
        if c not in df.columns:
            df[c] = 0
    return df[cm['features']]

def drift(df, th=3.0):
    st = json.loads((dd / 'train_stats.json').read_text(encoding='utf-8'))
    rs = {}
    for c, v in st.items():
        if c in df.columns:
            sd = v['std'] if v['std'] else 1.0
            z = abs(float(df[c].mean()) - v['mean']) / sd
            if z > th:
                rs[c] = round(z, 4)
    return rs

def load():
    global mm
    if mm is not None:
        return mm
    try:
        mm = mlflow.sklearn.load_model('models:/churn_model/Production')
    except Exception:
        mm = joblib.load(md / 'best_model.pkl')
    return mm

@app.get('/')
def home():
    return {'status': 'ok'}

@app.post('/predict')
def predict(x: dict):
    m = load()
    tb = prep(x)
    y = int(m.predict(tb)[0])
    p = float(m.predict_proba(tb)[0][1]) if hasattr(m, 'predict_proba') else float(y)
    dr = drift(tb)
    with (ld / 'predictions.jsonl').open('a', encoding='utf-8') as f:
        f.write(json.dumps({'time': datetime.utcnow().isoformat(), 'input': x, 'label': y, 'probability': p, 'drift': dr}, ensure_ascii=False) + '\n')
    return {'label': y, 'probability': p, 'drift': bool(dr), 'drift_features': dr}

if not (md / 'best_model.pkl').exists():
    print('run train notebook first')
else:
    def run_api():
        uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')
    threading.Thread(target=run_api, daemon=True).start()
    time.sleep(4)
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(8000)
    except Exception:
        print('API: http://127.0.0.1:8000')
    x = json.loads((md / 'sample_input.json').read_text(encoding='utf-8'))
    r = requests.post('http://127.0.0.1:8000/predict', json=x, timeout=30)
    print(r.json())
    print('curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d @models/sample_input.json')


C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


API: http://127.0.0.1:8000


{'label': 1, 'probability': 0.98, 'drift': False, 'drift_features': {}}
curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d @models/sample_input.json
